## Example 1: Compress and decompress with snappy.compress / snappy.decompress

In [ ]:
import snappy

# Example 1: Basic Snappy compression
print("Testing python-snappy basic compression:")

# Test data
data = b"Hello World! This is a test string for Snappy compression."

# Compress
compressed = snappy.compress(data)
print(f"✓ Original: {len(data)} bytes")
print(f"✓ Compressed: {len(compressed)} bytes")

# Decompress
decompressed = snappy.decompress(compressed)
assert isinstance(decompressed, bytes) and decompressed == data
print(f"✓ Decompressed: {len(decompressed)} bytes")
print(f"✓ Round-trip: {decompressed == data}")

# A str is encoded (UTF-8 by default) before compression; `decoding` turns the result back into a str
text = "Snappy handles text too: café"
assert snappy.decompress(snappy.compress(text), decoding="utf-8") == text
print("✓ str round-trip with decoding='utf-8'")

# Test with repetitive data
repeat_data = b"ABCD" * 100
compressed_repeat = snappy.compress(repeat_data)
assert len(compressed_repeat) < len(repeat_data)
ratio = len(compressed_repeat) / len(repeat_data)
print(f"✓ Repetitive data: {len(repeat_data)} -> {len(compressed_repeat)} bytes (ratio: {ratio:.2f})")

## Example 2: The raw block format and the framing (stream) format

In [ ]:
import snappy

# Example 2: Raw block format vs. the framing format
print("Testing python-snappy raw and framed formats:")

data = b"Raw compression test data " * 10

# snappy.compress produces a single raw Snappy block (no frame)
raw = snappy.compress(data)
print(f"✓ Raw: {len(data)} -> {len(raw)} bytes")

# StreamCompressor produces the framing format: a stream identifier chunk, then framed data chunks
framed = snappy.StreamCompressor().add_chunk(data)
assert framed.startswith(b"\xff\x06\x00\x00sNaPpY"), framed[:10]
print(f"✓ Framed: {len(data)} -> {len(framed)} bytes")

# Frame vs raw size difference
print(f"✓ Frame overhead: {len(framed) - len(raw)} bytes")

# Each format has its own decoder
assert snappy.decompress(raw) == data
assert snappy.StreamDecompressor().decompress(framed) == data
print("✓ Both formats decode back to the original data")

## Example 3: Validating input and handling errors

In [ ]:
import snappy

# Example 3: isValidCompressed and UncompressError
print("Testing python-snappy validation and errors:")

data = b"Test data for validation"
compressed = snappy.compress(data)

assert snappy.isValidCompressed(compressed)
print(f"✓ isValidCompressed(compressed data): {snappy.isValidCompressed(compressed)}")

garbage = b"\xff\xff\xff\xff this is not Snappy data"
assert not snappy.isValidCompressed(garbage)
print(f"✓ isValidCompressed(garbage): {snappy.isValidCompressed(garbage)}")

try:
    snappy.decompress(garbage)
except snappy.UncompressError as e:
    print(f"✓ decompress(garbage) raised {type(e).__name__}")
else:
    raise AssertionError("decompress() accepted data that is not Snappy-compressed")

## Example 4: Compressing file-like objects with stream_compress / stream_decompress

In [ ]:
import io
import snappy

# Example 4: Compress and decompress file-like objects
print("Testing python-snappy stream_compress / stream_decompress:")

# About 230 KB, so the stream is written as several 64 KiB blocks
data = b"A line of log output for streaming compression\n" * 5000
src, compressed = io.BytesIO(data), io.BytesIO()
snappy.stream_compress(src, compressed)
print(f"✓ Streamed {len(data)} -> {compressed.tell()} bytes")

# The output is in the framing format, so it starts with the stream identifier
compressed.seek(0)
assert snappy.StreamDecompressor.check_format(compressed)

compressed.seek(0)
restored = io.BytesIO()
snappy.stream_decompress(compressed, restored)
assert restored.getvalue() == data
print(f"✓ Decompressed stream matches the original: {len(restored.getvalue())} bytes")

## Example 5: Incremental decompression of a stream that arrives in pieces

In [ ]:
import snappy

# Example 5: Feed a framed stream to StreamDecompressor in pieces
print("Testing python-snappy incremental decompression:")

chunks = [b"first chunk of data " * 50, b"second chunk " * 80, b"third and last chunk " * 30]
compressor = snappy.StreamCompressor()
stream = b"".join(compressor.add_chunk(chunk) for chunk in chunks)
print(f"✓ Compressed {len(chunks)} chunks into a {len(stream)}-byte stream")

# Pieces of 16 bytes split chunk headers and chunk bodies at arbitrary points, as a network read would
piece_size = 16
decompressor = snappy.StreamDecompressor()
pieces = [stream[i:i + piece_size] for i in range(0, len(stream), piece_size)]
result = b"".join(decompressor.decompress(piece) for piece in pieces)
decompressor.flush()
assert result == b"".join(chunks)
print(f"✓ Reassembled {len(result)} bytes from {len(pieces)} pieces")